# KO vs host pseudo-bulk and chi-square screen

Builds KO vs host pseudo-bulk per lineage for the chimeras and runs the chi-square screen.

Cheng, Stok Ranen et al., bioRxiv, doi:10.64898/2026.05.14.725081.

Panels using it (each figure notebook calls `prep_single_cell()` from `scripts/single_cell_func.r` itself):
- **1i**: KO vs host lineage profile correlation (`Fig1.ipynb`)
- **2e**: DKO vs TKO changes against host (`Fig2.ipynb`)
- **ED 5a**: TKO and DKO vs host per lineage (`ExtData_RNA.ipynb`)
- **ED 8c**: TKO vs DKO changes, methylation-sensitive TSS genes (`Fig5.ipynb`)
- **Supp. Table 1**: chi-square screen (section: Chi-square screen; table written by `Supp_tables.ipynb`)

Notes:
- Run from the repository root.
- Caches go through `out_path()` (`paths.R`), which maps `data/output/...` to `git_test/data/output/...`.

## Initialization

In [ ]:
source("../paths.R")
setwd(embdnmt_dir)

In [ ]:
source(file.path(repo_dir, "scripts/single_cell_func.r"))

### KO and host pseudo-bulk per lineage

- Input: single cells from `data/MC_models/RNA_all.h5ad`; cell types from `data/ref/cell_annotation.csv`; lineages from `data/ref/annot_key.txt`.
- Per genotype and lineage, each embryo with > 2 KO and > 2 host cells adds its KO and host UMIs, downsampled to the smaller total (seed 19).
- Embryos are summed into `mko` and `mhost` (genotype x lineage).
- Cached in `data/output/mko.RDS`, `mhost.RDS`; recomputed only if missing or with `recompute = TRUE` (writes to `git_test/data/output/`).

In [ ]:
pb = load_ko_host_pseudobulk()
mko = pb$mko
mhost = pb$mhost

## Chi-square screen

Writes `data/output/chisq_screen.RDS`, used by `prep_single_cell()`, `plt_exp_atac_tss()` (Fig. 5c) and `Supp_tables.ipynb`. Not needed to reproduce the figures.

- Per gene, genotype and lineage: 2 x 2 chi-square of the gene's UMIs vs all other UMIs, KO vs host.
- Skips genes with < 20 UMIs (KO + host).
- FDR per lineage; `sig` is q < 0.01.
- Caveat: the filter lists Tet-TKO, but the cached matrices have no Tet-TKO columns, so only Dnmt-TKO and -DKO are tested.
- A re-run writes to `git_test/data/output/`; figure notebooks read it with `use_regenerated_inputs = TRUE` (`paths.R`).

In [ ]:
res = calc_chisq_screen(mko, mhost)

In [21]:
saveRDS(res, out_path("./data/output/chisq_screen.RDS"))